In [ ]:
#https://www.udemy.com/course/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/?referralCode=31C17C306A59601B8689

#MCP è molto utile se devi connettere molti sistemi enterprise (e.g.Google Drive, Notion, Slack, ect), AL POSTO DI SCIVVERE UN AGENTE CUSTOMIZZATO SU QUELLO, esponi semplicemente la funzione da utilizzare, cosi molti tools NON devono girare dentro agent, utile anche x modularità.

import subprocess   #x eseguire processi(programmi) esterni
import sys  #info sul python corrente

def ask_airbnb(query):
    code = f"""
    import asyncio  
    from airbnb_mcp import search
    asyncio.run(search("{query}"))
    """  #miniscript python, asyncio per eseguire codice asincrono (perche search() che importi è async!)
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True) #lancia un nuovo python ed esegui quel codice. sys.executable è il python interpreter corrente, -c significa "esegui il codice passato, come stringa", code è il code sritto sopra, capture_output=True cattura stdout/stderr invece di stamparli nel terminale, text=True restituisce testo stringa invece di bytes
    return result.stdout  #return output standard del subprocess

In [ ]:
query = "show me the best hotels in mumbai"
response = ask_airbnb(query)
print(response)

In [ ]:
######################################################AIRBNB MCP

from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage

from typing_extensions import TypedDict, Annotated
import operator

# Config
LLM_MODEL = "qwen3"
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=LLM_MODEL, 
    base_url=BASE_URL, 
    temperature=0)

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

##### MCP SETUP #####
# MCP GITHUB
# https://github.com/laxmimerit/MCP-Mastery-with-Claude-and-Langchain
# https://github.com/langchain-ai/langchain-mcp-adapters

from langchain_mcp_adapters.client import MultiServerMCPClient  #permette a langchain/langgraph di connettersi a mcpserver, SUPPORTA ANCHE MULTIPLI MCPSERVERS CONTEMPORANEAMENTE
import asyncio  #x async/await

async def get_tools():  #ASYNC!
    client = MultiServerMCPClient(  #crei mcp client
        {
            "airbnb": {  #NOME LOGICO SERVER 
                "command": "npx",  #per avviare il server mcp, usa questo comando terminale
                "args": ["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],  #argomenti passati a npx
                "transport": "stdio"  #Host ↔ server comunicano via stdin/stdout
            }
        }
    )
    tools = await client.get_tools()  #chiede a mcp server "quali tools hai disponibili?"
    print(f"Loaded {len(tools)} Tools")
    print(f"Tools Available: {tools}")
    return tools

async def agent_node(state: AgentState):
    tools = await get_tools()
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}

async def create_agent():
    tools = await get_tools()
    builder = StateGraph(AgentState)
    builder.add_node('agent', agent_node)
    builder.add_node('tools', ToolNode(tools))
    # add edges
    builder.add_edge(START, 'agent')
    builder.add_edge('tools', 'agent')
    builder.add_conditional_edges('agent', tools_condition)
    graph = builder.compile()
    return graph

async def search(query):
    agent = await create_agent()
    result = await agent.ainvoke({'messages': [HumanMessage(query)]})
    response = result['messages'][-1].content
    print(response)
    return response

if __name__=="__main__":
    query = "Show me premium hotels for party in Mumbai"
    asyncio.run(search(query))